سلول ۱ — مسیرها و پیدا کردن فایل‌ها

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(".")

DATA_PROC = PROJECT_ROOT / "Data_proc"
DATA_ML = PROJECT_ROOT / "Data_ml"
GRAPH_DIR = DATA_ML / "graph_dataset"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

def show_matches(pattern):
    print("\nPATTERN:", pattern)
    for p in PROJECT_ROOT.rglob(pattern):
        print(p)

show_matches("*pairs*.csv")
show_matches("*biogrid*.csv")
show_matches("*local*.csv")
show_matches("*scrna*.csv")

سلول ۲ — لود pair نهایی و feature نهایی

In [ ]:
PAIR_PATH_CANDIDATES = [
    DATA_PROC / "pairs" / "pairs_all_embedding_ready.csv",
    DATA_PROC / "pairs" / "pairs_all_model_ready.csv",
    DATA_PROC / "pairs" / "pairs_all_final.csv",
]

for p in PAIR_PATH_CANDIDATES:
    print(p, p.exists())

PAIR_PATH = [p for p in PAIR_PATH_CANDIDATES if p.exists()][0]

pairs = pd.read_csv(PAIR_PATH, dtype=str, low_memory=False)
pairs["label"] = pairs["label"].astype(int)

print("PAIR_PATH:", PAIR_PATH)
print("pairs:", pairs.shape)
print(pairs.columns.tolist())
display(pairs.head())
print(pairs["label"].value_counts())

سلول ۳ — ساخت node table

In [ ]:
enz_nodes = pairs[["enz_ac", "enz_gene", "enzyme_class"]].copy()
enz_nodes = enz_nodes.rename(columns={
    "enz_ac": "uniprot_ac",
    "enz_gene": "gene",
})

sub_nodes = pairs[["sub_ac", "sub_gene"]].copy()
sub_nodes = sub_nodes.rename(columns={
    "sub_ac": "uniprot_ac",
    "sub_gene": "gene",
})
sub_nodes["enzyme_class"] = "SUBSTRATE"

nodes = pd.concat([enz_nodes, sub_nodes], ignore_index=True)
nodes = nodes.drop_duplicates("uniprot_ac").reset_index(drop=True)

nodes["node_id"] = np.arange(len(nodes))

node_map = dict(zip(nodes["uniprot_ac"], nodes["node_id"]))

nodes["is_e3"] = (nodes["enzyme_class"] == "E3").astype(int)
nodes["is_dub"] = (nodes["enzyme_class"] == "DUB").astype(int)
nodes["is_substrate"] = nodes["uniprot_ac"].isin(pairs["sub_ac"]).astype(int)

print("nodes:", nodes.shape)
display(nodes.head())

nodes.to_csv(GRAPH_DIR / "graph_nodes.csv", index=False)

سلول ۴ — ساخت edgeهای اصلی برای link prediction

In [ ]:
interaction_edges = pairs.copy()

interaction_edges["src"] = interaction_edges["enz_ac"].map(node_map)
interaction_edges["dst"] = interaction_edges["sub_ac"].map(node_map)

interaction_edges = interaction_edges.dropna(subset=["src", "dst"]).copy()
interaction_edges["src"] = interaction_edges["src"].astype(int)
interaction_edges["dst"] = interaction_edges["dst"].astype(int)

interaction_edges["edge_type"] = "enzyme_substrate"
interaction_edges["edge_label"] = interaction_edges["label"].astype(int)

edge_cols = [
    "pair_id", "group_id", "enzyme_class",
    "enz_ac", "sub_ac", "enz_gene", "sub_gene",
    "src", "dst", "edge_type", "edge_label"
]

interaction_edges = interaction_edges[edge_cols].copy()

print("interaction_edges:", interaction_edges.shape)
print(interaction_edges["edge_label"].value_counts())
display(interaction_edges.head())

interaction_edges.to_csv(GRAPH_DIR / "interaction_edges_labeled.csv", index=False)

سلول ۵ — node features از بهترین embedding

برای GNN، node feature باید برای هر پروتئین یک بردار باشد. فعلاً بهترین و سبک‌ترین انتخاب:

In [ ]:
EMB_MODEL = "esm2_t33_650M_UR50D"
EMB_DIR = DATA_ML / "pair_features"  # نه؛ این feature جفتی است

ما node embedding خام را باید از Data_feat بخوانیم:

In [ ]:
EMB_MODEL = "esm2_t33_650M_UR50D"
EMB_DIR = DATA_ML / "pair_features"  # نه؛ این feature جفتی است

سلول ساخت node features

In [ ]:
import numpy as np
from pathlib import Path

DATA_FEAT = PROJECT_ROOT / "Data_feat"
NODE_EMB_MODEL = "esm2_t33_650M_UR50D"

node_feature_list = []
missing = []

for acc in nodes["uniprot_ac"].astype(str).tolist():
    emb_path = (
        DATA_FEAT
        / "per_sequence"
        / NODE_EMB_MODEL
        / "proteins"
        / f"{acc}.npy"
    )
    
    if not emb_path.exists():
        missing.append(acc)
        node_feature_list.append(None)
    else:
        x = np.load(emb_path).astype(np.float32)
        node_feature_list.append(x)

print("missing:", len(missing))
print(missing[:20])

In [ ]:
assert len(missing) == 0, f"Missing node embeddings: {missing[:20]}"

node_features = np.vstack(node_feature_list).astype(np.float32)

print("node_features:", node_features.shape)

np.save(
    GRAPH_DIR / "node_features_esm650.npy",
    node_features
)

سلول ۶ — QC گراف اولیه

In [ ]:
qc = {
    "n_nodes": len(nodes),
    "n_interaction_edges": len(interaction_edges),
    "n_positive_edges": int((interaction_edges["edge_label"] == 1).sum()),
    "n_negative_edges": int((interaction_edges["edge_label"] == 0).sum()),
    "n_unique_src": interaction_edges["src"].nunique(),
    "n_unique_dst": interaction_edges["dst"].nunique(),
    "node_feature_shape": str(node_features.shape),
    "missing_node_embeddings": len(missing),
    "duplicate_pair_id": int(interaction_edges.duplicated("pair_id").sum()),
}

qc_df = pd.DataFrame([qc])
display(qc_df)

qc_df.to_csv(GRAPH_DIR / "graph_dataset_qc.csv", index=False)

In [ ]:
for x in [
    "nodes",
    "interaction_edges",
    "X_nodes",
    "missing"
]:
    print(x, x in globals())

In [ ]:
print(len(nodes) if "nodes" in globals() else "NO_NODES")

print(
    len(interaction_edges)
    if "interaction_edges" in globals()
    else "NO_EDGES"
)

فایل BioGRID

In [ ]:
ppi = pd.read_csv(
"./Data_interim/biogrid/biogrid_human_physical_edges.csv"
)

print(ppi.shape)
print(ppi.columns.tolist())
display(ppi.head())

فایل Localization

In [ ]:
loc = pd.read_csv(
"./Data_interim/localization/uniprot_localization_normalized.csv"
)

print(loc.shape)
print(loc.columns.tolist())
display(loc.head())

سلول ۷ — ساخت PPI edges از BioGRID

In [ ]:
PPI_PATH = PROJECT_ROOT / "Data_interim" / "biogrid" / "biogrid_human_physical_edges.csv"

ppi = pd.read_csv(PPI_PATH, dtype=str, low_memory=False)

print("ppi raw:", ppi.shape)
print(ppi.columns.tolist())
display(ppi.head())

ppi_edges = ppi.copy()

ppi_edges = ppi_edges[
    ppi_edges["geneA"].isin(nodes["gene"].dropna())
    &
    ppi_edges["geneB"].isin(nodes["gene"].dropna())
].copy()

gene_to_acc = (
    nodes
    .dropna(subset=["gene"])
    .drop_duplicates("gene")
    .set_index("gene")["uniprot_ac"]
    .to_dict()
)

ppi_edges["src_ac"] = ppi_edges["geneA"].map(gene_to_acc)
ppi_edges["dst_ac"] = ppi_edges["geneB"].map(gene_to_acc)

ppi_edges = ppi_edges.dropna(subset=["src_ac", "dst_ac"]).copy()

ppi_edges["src"] = ppi_edges["src_ac"].map(node_map)
ppi_edges["dst"] = ppi_edges["dst_ac"].map(node_map)

ppi_edges = ppi_edges.dropna(subset=["src", "dst"]).copy()

ppi_edges["src"] = ppi_edges["src"].astype(int)
ppi_edges["dst"] = ppi_edges["dst"].astype(int)

ppi_edges = ppi_edges[ppi_edges["src"] != ppi_edges["dst"]].copy()

ppi_edges["edge_type"] = "ppi"
ppi_edges["edge_label"] = -1

ppi_edges["edge_id"] = (
    ppi_edges[["src", "dst"]]
    .min(axis=1)
    .astype(str)
    +
    "|"
    +
    ppi_edges[["src", "dst"]]
    .max(axis=1)
    .astype(str)
)

ppi_edges = ppi_edges.drop_duplicates("edge_id").reset_index(drop=True)

ppi_edges = ppi_edges[[
    "src",
    "dst",
    "src_ac",
    "dst_ac",
    "geneA",
    "geneB",
    "edge_type",
    "edge_label",
    "edge_id",
]]

print("ppi_edges:", ppi_edges.shape)
display(ppi_edges.head())

ppi_edges.to_csv(GRAPH_DIR / "ppi_edges.csv", index=False)

سلول ۸ — ساخت co-localization edges

این edgeها بین پروتئین‌هایی ساخته می‌شوند که حداقل یک compartment مشترک دارند.

In [ ]:
LOC_PATH = PROJECT_ROOT / "Data_interim" / "localization" / "uniprot_localization_normalized.csv"

loc = pd.read_csv(LOC_PATH, dtype=str, low_memory=False)

print("loc raw:", loc.shape)
print(loc.columns.tolist())
display(loc.head())

loc_proj = loc[
    loc["uniprot_ac"].isin(nodes["uniprot_ac"])
].dropna(subset=["uniprot_ac", "compartment"]).copy()

loc_proj = loc_proj[["uniprot_ac", "compartment"]].drop_duplicates()

loc_edges_rows = []

for compartment, g in loc_proj.groupby("compartment"):
    accs = sorted(g["uniprot_ac"].unique())
    
    # جلوگیری از انفجار edgeها
    if len(accs) < 2:
        continue
    
    if len(accs) > 500:
        print("Skipping very broad compartment:", compartment, len(accs))
        continue
    
    node_ids = [node_map[a] for a in accs if a in node_map]
    
    for i in range(len(node_ids)):
        for j in range(i + 1, len(node_ids)):
            loc_edges_rows.append({
                "src": node_ids[i],
                "dst": node_ids[j],
                "src_ac": accs[i],
                "dst_ac": accs[j],
                "compartment": compartment,
                "edge_type": "co_localized",
                "edge_label": -1,
            })

loc_edges = pd.DataFrame(loc_edges_rows)

if len(loc_edges):
    loc_edges["edge_id"] = (
        loc_edges[["src", "dst"]]
        .min(axis=1)
        .astype(str)
        +
        "|"
        +
        loc_edges[["src", "dst"]]
        .max(axis=1)
        .astype(str)
        +
        "|"
        +
        loc_edges["compartment"].astype(str)
    )
    
    loc_edges = loc_edges.drop_duplicates("edge_id").reset_index(drop=True)

print("loc_edges:", loc_edges.shape)
display(loc_edges.head())

loc_edges.to_csv(GRAPH_DIR / "colocalization_edges.csv", index=False)

سلول ۹ — ساخت coexpression edges از ستون‌های scRNA داخل pairs


اینجا فقط pairهایی را edge کمکی می‌کنیم که scrna_coexpr_flag == True دارند. این‌ها الزاماً مثبت نیستند؛ فقط context edge هستند.

In [ ]:
scrna_edges = pairs.copy()

if "scrna_coexpr_flag" in scrna_edges.columns:
    flag = scrna_edges["scrna_coexpr_flag"].astype(str).str.lower().isin(["true", "1", "yes"])
else:
    flag = pd.Series(False, index=scrna_edges.index)

scrna_edges = scrna_edges[flag].copy()

scrna_edges["src"] = scrna_edges["enz_ac"].map(node_map)
scrna_edges["dst"] = scrna_edges["sub_ac"].map(node_map)

scrna_edges = scrna_edges.dropna(subset=["src", "dst"]).copy()

scrna_edges["src"] = scrna_edges["src"].astype(int)
scrna_edges["dst"] = scrna_edges["dst"].astype(int)

scrna_edges["src_ac"] = scrna_edges["enz_ac"]
scrna_edges["dst_ac"] = scrna_edges["sub_ac"]

scrna_edges["edge_type"] = "coexpressed"
scrna_edges["edge_label"] = -1

scrna_edges["edge_id"] = (
    scrna_edges["src"].astype(str)
    +
    "|"
    +
    scrna_edges["dst"].astype(str)
    +
    "|coexpressed"
)

keep_cols = [
    "src",
    "dst",
    "src_ac",
    "dst_ac",
    "enz_gene",
    "sub_gene",
    "edge_type",
    "edge_label",
    "edge_id",
]

extra_cols = [
    "scrna_coexpr_crc",
    "scrna_coexpr_lihc",
    "scrna_max_min_expr",
    "scrna_best_cancer",
    "scrna_best_celltype",
]

keep_cols = keep_cols + [c for c in extra_cols if c in scrna_edges.columns]

scrna_edges = scrna_edges[keep_cols].drop_duplicates("edge_id").reset_index(drop=True)

print("scrna_edges:", scrna_edges.shape)
display(scrna_edges.head())

scrna_edges.to_csv(GRAPH_DIR / "coexpression_edges.csv", index=False)

سلول ۱۰ — ترکیب همه edgeها

In [ ]:
interaction_context = interaction_edges.copy()
interaction_context["src_ac"] = interaction_context["enz_ac"]
interaction_context["dst_ac"] = interaction_context["sub_ac"]
interaction_context["edge_id"] = interaction_context["pair_id"]

interaction_context = interaction_context[[
    "src",
    "dst",
    "src_ac",
    "dst_ac",
    "edge_type",
    "edge_label",
    "edge_id",
]]

edge_tables = [interaction_context]

for df in [ppi_edges, loc_edges, scrna_edges]:
    if df is not None and len(df):
        tmp = df[[
            "src",
            "dst",
            "src_ac",
            "dst_ac",
            "edge_type",
            "edge_label",
            "edge_id",
        ]].copy()
        edge_tables.append(tmp)

graph_edges_all = pd.concat(edge_tables, ignore_index=True)

print("graph_edges_all:", graph_edges_all.shape)
print(graph_edges_all["edge_type"].value_counts())
display(graph_edges_all.head())

graph_edges_all.to_csv(GRAPH_DIR / "graph_edges_all.csv", index=False)

سلول ۱۱ — QC نهایی edgeها

In [ ]:
graph_qc = {
    "n_nodes": len(nodes),
    "node_feature_shape": str(node_features.shape),
    "n_interaction_edges": len(interaction_edges),
    "n_ppi_edges": len(ppi_edges),
    "n_colocalization_edges": len(loc_edges),
    "n_coexpression_edges": len(scrna_edges),
    "n_all_edges": len(graph_edges_all),
    "n_positive_labeled_edges": int((interaction_edges["edge_label"] == 1).sum()),
    "n_negative_labeled_edges": int((interaction_edges["edge_label"] == 0).sum()),
    "missing_node_embeddings": len(missing),
    "duplicate_labeled_pair_id": int(interaction_edges.duplicated("pair_id").sum()),
}

graph_qc_df = pd.DataFrame([graph_qc])
display(graph_qc_df)

graph_qc_df.to_csv(GRAPH_DIR / "graph_full_dataset_qc.csv", index=False)

In [ ]:
import torch
print(torch.__version__)
print(torch.backends.mps.is_available())

In [ ]:
import sys
print(sys.version)

اینچا از cheminformatics اومدیم روی e3gnn

سلول ۱۲ — تست محیط جدید

In [ ]:
import sys
import torch
import torch_geometric

print(sys.version)
print("torch:", torch.__version__)
print("pyg:", torch_geometric.__version__)
print("mps:", torch.backends.mps.is_available())

سلول ۱۳ — لود گراف ذخیره‌شده

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch

PROJECT_ROOT = Path(".")
GRAPH_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset"

nodes = pd.read_csv(GRAPH_DIR / "graph_nodes.csv")
interaction_edges = pd.read_csv(GRAPH_DIR / "interaction_edges_labeled.csv")
graph_edges_all = pd.read_csv(GRAPH_DIR / "graph_edges_all.csv")
node_features = np.load(GRAPH_DIR / "node_features_esm650.npy")

print("nodes:", nodes.shape)
print("interaction_edges:", interaction_edges.shape)
print("graph_edges_all:", graph_edges_all.shape)
print("node_features:", node_features.shape)

display(nodes.head())
display(interaction_edges.head())
display(graph_edges_all["edge_type"].value_counts())

سلول ۱۴ — ساخت PyG Data برای GraphSAGE

In [ ]:
from torch_geometric.data import Data

x = torch.tensor(node_features, dtype=torch.float32)

# فقط edgeهای کمکی + interactionها برای message passing
edge_index = torch.tensor(
    graph_edges_all[["src", "dst"]].values.T,
    dtype=torch.long
)

# گراف را undirected می‌کنیم
edge_index_rev = edge_index[[1, 0], :]
edge_index = torch.cat([edge_index, edge_index_rev], dim=1)

# labeled edges برای link prediction
edge_label_index = torch.tensor(
    interaction_edges[["src", "dst"]].values.T,
    dtype=torch.long
)

edge_label = torch.tensor(
    interaction_edges["edge_label"].astype(int).values,
    dtype=torch.float32
)

data = Data(
    x=x,
    edge_index=edge_index,
    edge_label_index=edge_label_index,
    edge_label=edge_label,
)

print(data)
print("x:", data.x.shape)
print("edge_index:", data.edge_index.shape)
print("edge_label_index:", data.edge_label_index.shape)
print("edge_label:", data.edge_label.shape)
print("labels:", torch.bincount(data.edge_label.long()))

سلول ۱۵ — GroupKFold split مثل قبل

In [ ]:
from sklearn.model_selection import GroupKFold

groups = interaction_edges["group_id"].values
y = interaction_edges["edge_label"].astype(int).values

gkf = GroupKFold(n_splits=5)

fold_splits = []

for fold, (tr, te) in enumerate(gkf.split(interaction_edges, y, groups)):
    fold_splits.append((tr, te))
    print(
        fold,
        "train:", len(tr),
        "test:", len(te),
        "test pos:", y[te].sum(),
        "test neg:", len(te) - y[te].sum()
    )

سلول ۱۶ — مدل GraphSAGE Link Predictor

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGEEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, out_dim=128, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.gelu(x)

        return x


class LinkPredictor(nn.Module):
    def __init__(self, emb_dim=128, hidden_dim=128, dropout=0.3):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, z, edge_label_index):
        src = edge_label_index[0]
        dst = edge_label_index[1]

        z_src = z[src]
        z_dst = z[dst]

        h = torch.cat([
            z_src,
            z_dst,
            torch.abs(z_src - z_dst),
            z_src * z_dst,
        ], dim=1)

        return self.mlp(h).squeeze(-1)


class GraphSAGELinkModel(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, emb_dim=128, dropout=0.3):
        super().__init__()
        self.encoder = GraphSAGEEncoder(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=emb_dim,
            dropout=dropout,
        )
        self.predictor = LinkPredictor(
            emb_dim=emb_dim,
            hidden_dim=128,
            dropout=dropout,
        )

    def forward(self, x, edge_index, edge_label_index):
        z = self.encoder(x, edge_index)
        logits = self.predictor(z, edge_label_index)
        return logits

سلول ۱۷ — Training loop

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    brier_score_loss,
)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("DEVICE:", DEVICE)

def compute_metrics(y_true, prob):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    pred = (prob >= 0.5).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, prob),
        "pr_auc": average_precision_score(y_true, prob),
        "f1": f1_score(y_true, pred, zero_division=0),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "brier": brier_score_loss(y_true, prob),
    }


def train_graphsage_fold(train_idx, test_idx, fold, epochs=200, lr=1e-3, weight_decay=1e-4):
    model = GraphSAGELinkModel(
        in_dim=data.x.shape[1],
        hidden_dim=256,
        emb_dim=128,
        dropout=0.35,
    ).to(DEVICE)

    x_dev = data.x.to(DEVICE)
    edge_index_dev = data.edge_index.to(DEVICE)

    train_edge_index = data.edge_label_index[:, train_idx].to(DEVICE)
    train_y = data.edge_label[train_idx].to(DEVICE)

    test_edge_index = data.edge_label_index[:, test_idx].to(DEVICE)
    test_y = data.edge_label[test_idx].cpu().numpy()

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_pr = -1
    best_state = None
    best_epoch = -1
    patience = 25
    bad_epochs = 0

    history = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(
            x_dev,
            edge_index_dev,
            train_edge_index,
        )

        loss = criterion(logits, train_y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)

        optimizer.step()

        model.eval()
        with torch.no_grad():
            test_logits = model(
                x_dev,
                edge_index_dev,
                test_edge_index,
            )

            prob = torch.sigmoid(test_logits).detach().cpu().numpy()

        metrics = compute_metrics(test_y, prob)
        metrics["epoch"] = epoch
        metrics["train_loss"] = float(loss.detach().cpu().item())
        history.append(metrics)

        if metrics["pr_auc"] > best_pr:
            best_pr = metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 20 == 0:
            print(
                f"fold={fold} epoch={epoch} "
                f"loss={loss.item():.4f} "
                f"pr={metrics['pr_auc']:.4f} "
                f"roc={metrics['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping:", fold, epoch)
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        test_logits = model(
            x_dev,
            edge_index_dev,
            test_edge_index,
        )
        prob = torch.sigmoid(test_logits).detach().cpu().numpy()

    final_metrics = compute_metrics(test_y, prob)
    final_metrics["fold"] = fold
    final_metrics["best_epoch"] = best_epoch

    pred_df = interaction_edges.iloc[test_idx][[
        "pair_id", "group_id", "enzyme_class",
        "enz_ac", "sub_ac", "enz_gene", "sub_gene",
        "edge_label"
    ]].copy()

    pred_df["fold"] = fold
    pred_df["y_true"] = test_y
    pred_df["prob_graphsage"] = prob

    hist_df = pd.DataFrame(history)
    hist_df["fold"] = fold

    return final_metrics, pred_df, hist_df

سلول ۱۸ — اجرای GraphSAGE

In [ ]:
graphsage_rows = []
graphsage_preds = []
graphsage_histories = []

for fold, (tr, te) in enumerate(fold_splits):
    print("\n" + "=" * 100)
    print("GraphSAGE Fold:", fold)

    metrics, pred_df, hist_df = train_graphsage_fold(
        train_idx=tr,
        test_idx=te,
        fold=fold,
        epochs=200,
        lr=1e-3,
        weight_decay=1e-4,
    )

    graphsage_rows.append(metrics)
    graphsage_preds.append(pred_df)
    graphsage_histories.append(hist_df)

graphsage_metrics = pd.DataFrame(graphsage_rows)
graphsage_predictions = pd.concat(graphsage_preds, ignore_index=True)
graphsage_history = pd.concat(graphsage_histories, ignore_index=True)

display(graphsage_metrics)

graphsage_summary = (
    graphsage_metrics
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(graphsage_summary)

graphsage_metrics.to_csv(GRAPH_DIR / "graphsage_fold_metrics.csv", index=False)
graphsage_predictions.to_csv(GRAPH_DIR / "graphsage_fold_predictions.csv", index=False)
graphsage_history.to_csv(GRAPH_DIR / "graphsage_training_history.csv", index=False)
graphsage_summary.to_csv(GRAPH_DIR / "graphsage_summary.csv")

سلول ۱۹ — تعریف GAT Link Predictor

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GATConv


class GATEncoder(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        out_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()
        
        self.gat1 = GATConv(
            in_channels=in_dim,
            out_channels=hidden_dim,
            heads=heads,
            dropout=dropout,
            concat=True,
        )
        
        self.gat2 = GATConv(
            in_channels=hidden_dim * heads,
            out_channels=out_dim,
            heads=1,
            dropout=dropout,
            concat=False,
        )
        
        self.dropout = dropout
        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.norm2 = nn.LayerNorm(out_dim)

    def forward(self, x, edge_index):
        x = self.gat1(x, edge_index)
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = self.gat2(x, edge_index)
        x = self.norm2(x)
        x = F.gelu(x)
        
        return x


class GATLinkModel(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()
        
        self.encoder = GATEncoder(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=emb_dim,
            heads=heads,
            dropout=dropout,
        )
        
        self.predictor = LinkPredictor(
            emb_dim=emb_dim,
            hidden_dim=128,
            dropout=dropout,
        )

    def forward(self, x, edge_index, edge_label_index):
        z = self.encoder(x, edge_index)
        logits = self.predictor(z, edge_label_index)
        return logits

سلول ۲۰ — Training loop برای GAT

In [ ]:
def train_gat_fold(
    train_idx,
    test_idx,
    fold,
    epochs=200,
    lr=5e-4,
    weight_decay=1e-4,
):
    model = GATLinkModel(
        in_dim=data.x.shape[1],
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ).to(DEVICE)

    x_dev = data.x.to(DEVICE)
    edge_index_dev = data.edge_index.to(DEVICE)

    train_edge_index = data.edge_label_index[:, train_idx].to(DEVICE)
    train_y = data.edge_label[train_idx].to(DEVICE)

    test_edge_index = data.edge_label_index[:, test_idx].to(DEVICE)
    test_y = data.edge_label[test_idx].cpu().numpy()

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_pr = -1
    best_state = None
    best_epoch = -1
    patience = 30
    bad_epochs = 0

    history = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(
            x_dev,
            edge_index_dev,
            train_edge_index,
        )

        loss = criterion(logits, train_y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()

        model.eval()

        with torch.no_grad():
            test_logits = model(
                x_dev,
                edge_index_dev,
                test_edge_index,
            )
            prob = torch.sigmoid(test_logits).detach().cpu().numpy()

        metrics = compute_metrics(test_y, prob)
        metrics["epoch"] = epoch
        metrics["train_loss"] = float(loss.detach().cpu().item())

        history.append(metrics)

        if metrics["pr_auc"] > best_pr:
            best_pr = metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 20 == 0:
            print(
                f"fold={fold} epoch={epoch} "
                f"loss={loss.item():.4f} "
                f"pr={metrics['pr_auc']:.4f} "
                f"roc={metrics['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping:", fold, epoch)
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        test_logits = model(
            x_dev,
            edge_index_dev,
            test_edge_index,
        )
        prob = torch.sigmoid(test_logits).detach().cpu().numpy()

    final_metrics = compute_metrics(test_y, prob)
    final_metrics["fold"] = fold
    final_metrics["best_epoch"] = best_epoch

    pred_df = interaction_edges.iloc[test_idx][[
        "pair_id", "group_id", "enzyme_class",
        "enz_ac", "sub_ac", "enz_gene", "sub_gene",
        "edge_label"
    ]].copy()

    pred_df["fold"] = fold
    pred_df["y_true"] = test_y
    pred_df["prob_gat"] = prob

    hist_df = pd.DataFrame(history)
    hist_df["fold"] = fold

    return final_metrics, pred_df, hist_df

سلول ۲۱ — اجرای GAT

In [ ]:
gat_rows = []
gat_preds = []
gat_histories = []

for fold, (tr, te) in enumerate(fold_splits):
    print("\n" + "=" * 100)
    print("GAT Fold:", fold)

    metrics, pred_df, hist_df = train_gat_fold(
        train_idx=tr,
        test_idx=te,
        fold=fold,
        epochs=200,
        lr=5e-4,
        weight_decay=1e-4,
    )

    gat_rows.append(metrics)
    gat_preds.append(pred_df)
    gat_histories.append(hist_df)

gat_metrics = pd.DataFrame(gat_rows)
gat_predictions = pd.concat(gat_preds, ignore_index=True)
gat_history = pd.concat(gat_histories, ignore_index=True)

display(gat_metrics)

gat_summary = (
    gat_metrics
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(gat_summary)

gat_metrics.to_csv(GRAPH_DIR / "gat_fold_metrics.csv", index=False)
gat_predictions.to_csv(GRAPH_DIR / "gat_fold_predictions.csv", index=False)
gat_history.to_csv(GRAPH_DIR / "gat_training_history.csv", index=False)
gat_summary.to_csv(GRAPH_DIR / "gat_summary.csv")

In [ ]:
print("graphsage")
print(graphsage_pred.shape)
print(graphsage_pred["pair_id"].nunique())

print()

print("xgb")
print(xgb_pred.shape)
print(xgb_pred["pair_id"].nunique())

print()

print("nn")
print(nn_pred.shape)
print(nn_pred["pair_id"].nunique())

In [ ]:
g = set(graphsage_pred["pair_id"])
x = set(xgb_pred["pair_id"])
n = set(nn_pred["pair_id"])

print("graph ∩ xgb:", len(g & x))
print("graph ∩ nn :", len(g & n))
print("all three :", len(g & x & n))

In [ ]:
print(graphsage_pred.columns.tolist())
print()
print(xgb_pred.columns.tolist())
print()
print(nn_pred.columns.tolist())

Graph Ensemble

سلول ۲۲ — لود predictionها

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    brier_score_loss,
)
from sklearn.linear_model import LogisticRegression

GRAPH_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset"
NN_RESULT_DIR = PROJECT_ROOT / "Data_ml" / "neural_pair_scorer_results"

graphsage_pred = pd.read_csv(GRAPH_DIR / "graphsage_fold_predictions.csv", dtype=str, low_memory=False)

xgb_pred = pd.read_csv(
    NN_RESULT_DIR / "xgb_best_fusion_fold_predictions_for_ensemble.csv",
    dtype=str,
    low_memory=False,
)

nn_pred = pd.read_csv(
    NN_RESULT_DIR / "nn_focal_fold_predictions.csv",
    dtype=str,
    low_memory=False,
)

print("graphsage:", graphsage_pred.shape)
print("xgb:", xgb_pred.shape)
print("nn:", nn_pred.shape)

display(graphsage_pred.head())
display(xgb_pred.head())
display(nn_pred.head())

سلول ۲۳ — یکسان‌سازی ستون‌ها و merge

In [ ]:
def clean_fold_column(df, col="fold"):
    df[col] = (
        pd.to_numeric(df[col], errors="coerce")
        .astype("Int64")
        .astype(int)
    )
    return df

graphsage_pred = clean_fold_column(graphsage_pred, "fold")
xgb_pred = clean_fold_column(xgb_pred, "fold")
nn_pred = clean_fold_column(nn_pred, "fold")

graphsage_pred["y_true"] = pd.to_numeric(graphsage_pred["y_true"], errors="coerce").astype(int)
graphsage_pred["prob_graphsage"] = pd.to_numeric(graphsage_pred["prob_graphsage"], errors="coerce")

xgb_pred["prob_xgb"] = pd.to_numeric(xgb_pred["prob_xgb"], errors="coerce")
nn_pred["prob_nn"] = pd.to_numeric(nn_pred["prob"], errors="coerce")

ens_graph_df = graphsage_pred[[
    "pair_id",
    "group_id",
    "enzyme_class",
    "enz_ac",
    "sub_ac",
    "enz_gene",
    "sub_gene",
    "edge_label",
    "fold",
    "y_true",
    "prob_graphsage",
]].copy()

ens_graph_df = ens_graph_df.merge(
    xgb_pred[["pair_id", "prob_xgb"]],
    on="pair_id",
    how="inner",
)

ens_graph_df = ens_graph_df.merge(
    nn_pred[["pair_id", "prob_nn"]],
    on="pair_id",
    how="inner",
)

print("merged:", ens_graph_df.shape)
print("unique pairs:", ens_graph_df["pair_id"].nunique())

print("missing values:")
print(
    ens_graph_df[
        ["prob_graphsage", "prob_xgb", "prob_nn", "y_true"]
    ].isna().sum()
)

assert ens_graph_df.shape[0] == 6196
assert ens_graph_df["pair_id"].nunique() == 6196

display(ens_graph_df.head())

سلول ۲۴ — تابع ارزیابی

In [ ]:
def evaluate_probs_by_fold(df, prob_col):
    rows = []

    for fold, g in df.groupby("fold"):
        y_true = g["y_true"].astype(int).values
        prob = g[prob_col].astype(float).values
        pred = (prob >= 0.5).astype(int)

        rows.append({
            "fold": fold,
            "roc_auc": roc_auc_score(y_true, prob),
            "pr_auc": average_precision_score(y_true, prob),
            "f1": f1_score(y_true, pred, zero_division=0),
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "brier": brier_score_loss(y_true, prob),
        })

    return pd.DataFrame(rows)

سلول ۲۵ — Weighted Ensemble: GraphSAGE + XGB + NN

In [ ]:
weight_sets = [
    {"name": "graph_only", "w_graph": 1.0, "w_xgb": 0.0, "w_nn": 0.0},
    {"name": "graph0.8_xgb0.2", "w_graph": 0.8, "w_xgb": 0.2, "w_nn": 0.0},
    {"name": "graph0.7_xgb0.3", "w_graph": 0.7, "w_xgb": 0.3, "w_nn": 0.0},
    {"name": "graph0.6_xgb0.4", "w_graph": 0.6, "w_xgb": 0.4, "w_nn": 0.0},
    {"name": "graph0.7_nn0.3", "w_graph": 0.7, "w_xgb": 0.0, "w_nn": 0.3},
    {"name": "graph0.6_nn0.4", "w_graph": 0.6, "w_xgb": 0.0, "w_nn": 0.4},
    {"name": "graph0.6_xgb0.2_nn0.2", "w_graph": 0.6, "w_xgb": 0.2, "w_nn": 0.2},
    {"name": "graph0.5_xgb0.25_nn0.25", "w_graph": 0.5, "w_xgb": 0.25, "w_nn": 0.25},
    {"name": "graph0.4_xgb0.3_nn0.3", "w_graph": 0.4, "w_xgb": 0.3, "w_nn": 0.3},
]

weighted_rows = []
weighted_fold_tables = []

for ws in weight_sets:
    col = "prob_" + ws["name"]

    ens_graph_df[col] = (
        ws["w_graph"] * ens_graph_df["prob_graphsage"]
        + ws["w_xgb"] * ens_graph_df["prob_xgb"]
        + ws["w_nn"] * ens_graph_df["prob_nn"]
    )

    fold_metrics = evaluate_probs_by_fold(ens_graph_df, col)
    fold_metrics["model"] = ws["name"]
    fold_metrics["w_graph"] = ws["w_graph"]
    fold_metrics["w_xgb"] = ws["w_xgb"]
    fold_metrics["w_nn"] = ws["w_nn"]

    weighted_fold_tables.append(fold_metrics)

    row = {
        "model": ws["name"],
        "w_graph": ws["w_graph"],
        "w_xgb": ws["w_xgb"],
        "w_nn": ws["w_nn"],
    }

    for m in ["roc_auc", "pr_auc", "f1", "accuracy", "precision", "recall", "brier"]:
        row[f"{m}_mean"] = fold_metrics[m].mean()
        row[f"{m}_std"] = fold_metrics[m].std()

    weighted_rows.append(row)

graph_weighted_ensemble_results = pd.DataFrame(weighted_rows).sort_values(
    "pr_auc_mean",
    ascending=False,
)

graph_weighted_ensemble_folds = pd.concat(weighted_fold_tables, ignore_index=True)

display(graph_weighted_ensemble_results)

graph_weighted_ensemble_results.to_csv(
    GRAPH_DIR / "graph_weighted_ensemble_results.csv",
    index=False,
)

graph_weighted_ensemble_folds.to_csv(
    GRAPH_DIR / "graph_weighted_ensemble_fold_results.csv",
    index=False,
)

سلول ۲۶ — Stacking: Logistic Regression روی GraphSAGE + XGB + NN

In [ ]:
stack_rows = []
stack_pred_rows = []

for test_fold in sorted(ens_graph_df["fold"].unique()):
    train_stack = ens_graph_df[ens_graph_df["fold"] != test_fold].copy()
    test_stack = ens_graph_df[ens_graph_df["fold"] == test_fold].copy()

    X_train_stack = train_stack[["prob_graphsage", "prob_xgb", "prob_nn"]].values
    y_train_stack = train_stack["y_true"].astype(int).values

    X_test_stack = test_stack[["prob_graphsage", "prob_xgb", "prob_nn"]].values
    y_test_stack = test_stack["y_true"].astype(int).values

    meta_model = LogisticRegression(
        solver="liblinear",
        random_state=42,
    )

    meta_model.fit(X_train_stack, y_train_stack)

    prob_stack = meta_model.predict_proba(X_test_stack)[:, 1]
    pred_stack = (prob_stack >= 0.5).astype(int)

    stack_rows.append({
        "fold": test_fold,
        "roc_auc": roc_auc_score(y_test_stack, prob_stack),
        "pr_auc": average_precision_score(y_test_stack, prob_stack),
        "f1": f1_score(y_test_stack, pred_stack, zero_division=0),
        "accuracy": accuracy_score(y_test_stack, pred_stack),
        "precision": precision_score(y_test_stack, pred_stack, zero_division=0),
        "recall": recall_score(y_test_stack, pred_stack),
        "brier": brier_score_loss(y_test_stack, prob_stack),
        "coef_graphsage": meta_model.coef_[0][0],
        "coef_xgb": meta_model.coef_[0][1],
        "coef_nn": meta_model.coef_[0][2],
        "intercept": meta_model.intercept_[0],
    })

    tmp = test_stack[[
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "edge_label",
        "fold",
        "y_true",
        "prob_graphsage",
        "prob_xgb",
        "prob_nn",
    ]].copy()

    tmp["prob_graph_stack"] = prob_stack
    stack_pred_rows.append(tmp)

graph_stack_metrics = pd.DataFrame(stack_rows)
graph_stack_predictions = pd.concat(stack_pred_rows, ignore_index=True)

display(graph_stack_metrics)

graph_stack_summary = (
    graph_stack_metrics
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(graph_stack_summary)

graph_stack_metrics.to_csv(
    GRAPH_DIR / "graph_stacking_fold_metrics.csv",
    index=False,
)

graph_stack_predictions.to_csv(
    GRAPH_DIR / "graph_stacking_predictions.csv",
    index=False,
)

graph_stack_summary.to_csv(
    GRAPH_DIR / "graph_stacking_summary.csv",
)

سلول ۲۷ — جدول نهایی روز ۱۲

In [ ]:
final_rows = []

for name, path in [
    ("GraphSAGE", GRAPH_DIR / "graphsage_fold_metrics.csv"),
    ("GAT", GRAPH_DIR / "gat_fold_metrics.csv"),
]:
    df = pd.read_csv(path)
    row = {"model": name}
    for m in ["roc_auc", "pr_auc", "f1", "accuracy", "precision", "recall", "brier"]:
        row[f"{m}_mean"] = df[m].mean()
        row[f"{m}_std"] = df[m].std()
    final_rows.append(row)

for _, r in graph_weighted_ensemble_results.iterrows():
    final_rows.append({
        "model": "weighted_" + r["model"],
        "roc_auc_mean": r["roc_auc_mean"],
        "roc_auc_std": r["roc_auc_std"],
        "pr_auc_mean": r["pr_auc_mean"],
        "pr_auc_std": r["pr_auc_std"],
        "f1_mean": r["f1_mean"],
        "f1_std": r["f1_std"],
        "accuracy_mean": r["accuracy_mean"],
        "accuracy_std": r["accuracy_std"],
        "precision_mean": r["precision_mean"],
        "precision_std": r["precision_std"],
        "recall_mean": r["recall_mean"],
        "recall_std": r["recall_std"],
        "brier_mean": r["brier_mean"],
        "brier_std": r["brier_std"],
    })

row = {"model": "Stacking_GraphSAGE_XGB_NN"}
for m in ["roc_auc", "pr_auc", "f1", "accuracy", "precision", "recall", "brier"]:
    row[f"{m}_mean"] = graph_stack_metrics[m].mean()
    row[f"{m}_std"] = graph_stack_metrics[m].std()
final_rows.append(row)

day12_graph_final_leaderboard = pd.DataFrame(final_rows).sort_values(
    "pr_auc_mean",
    ascending=False,
)

display(day12_graph_final_leaderboard)

day12_graph_final_leaderboard.to_csv(
    GRAPH_DIR / "day12_graph_final_leaderboard.csv",
    index=False,
)

In [ ]:
!pip install torch-geometric

یاداوریه مسیرها

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch

PROJECT_ROOT = Path(".")
GRAPH_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset"

nodes = pd.read_csv(GRAPH_DIR / "graph_nodes.csv")
interaction_edges = pd.read_csv(GRAPH_DIR / "interaction_edges_labeled.csv")
ppi_edges = pd.read_csv(GRAPH_DIR / "ppi_edges.csv")
loc_edges = pd.read_csv(GRAPH_DIR / "colocalization_edges.csv")
graph_edges_all = pd.read_csv(GRAPH_DIR / "graph_edges_all.csv")
node_features = np.load(GRAPH_DIR / "node_features_esm650.npy")

print("nodes:", nodes.shape)
print("interaction_edges:", interaction_edges.shape)
print("ppi_edges:", ppi_edges.shape)
print("loc_edges:", loc_edges.shape)
print("graph_edges_all:", graph_edges_all.shape)
print("node_features:", node_features.shape)

سلول ۲۸ — ساخت HeteroData

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("DEVICE:", DEVICE)

hetero_data = HeteroData()

hetero_data["protein"].x = torch.tensor(
    node_features,
    dtype=torch.float32
)

def edge_index_from_df(df):
    return torch.tensor(
        df[["src", "dst"]].values.T,
        dtype=torch.long
    )

# interaction edges
interaction_posneg = interaction_edges.copy()
interaction_edge_index = edge_index_from_df(interaction_posneg)

hetero_data["protein", "enzyme_substrate", "protein"].edge_index = interaction_edge_index

# ppi edges
ppi_edge_index = edge_index_from_df(ppi_edges)
ppi_edge_index = torch.cat(
    [ppi_edge_index, ppi_edge_index[[1, 0], :]],
    dim=1
)
hetero_data["protein", "ppi", "protein"].edge_index = ppi_edge_index

# co-localization edges
loc_edge_index = edge_index_from_df(loc_edges)
loc_edge_index = torch.cat(
    [loc_edge_index, loc_edge_index[[1, 0], :]],
    dim=1
)
hetero_data["protein", "co_localized", "protein"].edge_index = loc_edge_index

# labeled edges for prediction
hetero_data["protein", "predicts", "protein"].edge_label_index = torch.tensor(
    interaction_edges[["src", "dst"]].values.T,
    dtype=torch.long
)

hetero_data["protein", "predicts", "protein"].edge_label = torch.tensor(
    interaction_edges["edge_label"].astype(int).values,
    dtype=torch.float32
)

print(hetero_data)

سلول ۲۹ — مدل HeteroGraphSAGE

In [ ]:
class HeteroGraphSAGEEncoder(nn.Module):
    def __init__(self, hidden_dim=256, out_dim=128, dropout=0.35):
        super().__init__()

        self.lin_in = nn.Linear(node_features.shape[1], hidden_dim)

        self.conv1 = HeteroConv(
            {
                ("protein", "enzyme_substrate", "protein"): SAGEConv((-1, -1), hidden_dim),
                ("protein", "ppi", "protein"): SAGEConv((-1, -1), hidden_dim),
                ("protein", "co_localized", "protein"): SAGEConv((-1, -1), hidden_dim),
            },
            aggr="sum",
        )

        self.conv2 = HeteroConv(
            {
                ("protein", "enzyme_substrate", "protein"): SAGEConv((-1, -1), out_dim),
                ("protein", "ppi", "protein"): SAGEConv((-1, -1), out_dim),
                ("protein", "co_localized", "protein"): SAGEConv((-1, -1), out_dim),
            },
            aggr="sum",
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(out_dim)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict):
        x = x_dict["protein"]
        x = self.lin_in(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.conv1(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.conv2(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm2(x)
        x = F.gelu(x)

        return {"protein": x}


class HeteroLinkPredictor(nn.Module):
    def __init__(self, emb_dim=128, hidden_dim=128, dropout=0.35):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, z_dict, edge_label_index):
        z = z_dict["protein"]

        src = edge_label_index[0]
        dst = edge_label_index[1]

        z_src = z[src]
        z_dst = z[dst]

        h = torch.cat(
            [
                z_src,
                z_dst,
                torch.abs(z_src - z_dst),
                z_src * z_dst,
            ],
            dim=1,
        )

        return self.mlp(h).squeeze(-1)


class HeteroGraphSAGELinkModel(nn.Module):
    def __init__(self, hidden_dim=256, emb_dim=128, dropout=0.35):
        super().__init__()

        self.encoder = HeteroGraphSAGEEncoder(
            hidden_dim=hidden_dim,
            out_dim=emb_dim,
            dropout=dropout,
        )

        self.predictor = HeteroLinkPredictor(
            emb_dim=emb_dim,
            hidden_dim=128,
            dropout=dropout,
        )

    def forward(self, data, edge_label_index):
        z_dict = self.encoder(
            data.x_dict,
            data.edge_index_dict,
        )

        logits = self.predictor(
            z_dict,
            edge_label_index,
        )

        return logits

سلول ۳۰ — انتقال HeteroData به دستگاه

In [ ]:
hetero_data = hetero_data.to(DEVICE)

print(hetero_data)

سلول ۳۱ — Training loop برای HeteroGNN

In [ ]:
def train_hetero_fold(
    train_idx,
    test_idx,
    fold,
    epochs=200,
    lr=7e-4,
    weight_decay=1e-4,
):
    model = HeteroGraphSAGELinkModel(
        hidden_dim=256,
        emb_dim=128,
        dropout=0.35,
    ).to(DEVICE)

    edge_label_index_all = hetero_data["protein", "predicts", "protein"].edge_label_index
    edge_label_all = hetero_data["protein", "predicts", "protein"].edge_label

    train_edge_index = edge_label_index_all[:, train_idx].to(DEVICE)
    train_y = edge_label_all[train_idx].to(DEVICE)

    test_edge_index = edge_label_index_all[:, test_idx].to(DEVICE)
    test_y = edge_label_all[test_idx].detach().cpu().numpy()

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_pr = -1
    best_state = None
    best_epoch = -1
    patience = 30
    bad_epochs = 0

    history = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(
            hetero_data,
            train_edge_index,
        )

        loss = criterion(logits, train_y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()

        model.eval()

        with torch.no_grad():
            test_logits = model(
                hetero_data,
                test_edge_index,
            )

            prob = torch.sigmoid(test_logits).detach().cpu().numpy()

        metrics = compute_metrics(test_y, prob)
        metrics["epoch"] = epoch
        metrics["train_loss"] = float(loss.detach().cpu().item())

        history.append(metrics)

        if metrics["pr_auc"] > best_pr:
            best_pr = metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 20 == 0:
            print(
                f"fold={fold} epoch={epoch} "
                f"loss={loss.item():.4f} "
                f"pr={metrics['pr_auc']:.4f} "
                f"roc={metrics['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping:", fold, epoch)
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        test_logits = model(
            hetero_data,
            test_edge_index,
        )
        prob = torch.sigmoid(test_logits).detach().cpu().numpy()

    final_metrics = compute_metrics(test_y, prob)
    final_metrics["fold"] = fold
    final_metrics["best_epoch"] = best_epoch

    pred_df = interaction_edges.iloc[test_idx][[
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "edge_label",
    ]].copy()

    pred_df["fold"] = fold
    pred_df["y_true"] = test_y
    pred_df["prob_hetero_graphsage"] = prob

    hist_df = pd.DataFrame(history)
    hist_df["fold"] = fold

    return final_metrics, pred_df, hist_df

سلول ۳۲ — اجرای Heterogeneous GNN

In [ ]:
hetero_rows = []
hetero_preds = []
hetero_histories = []

for fold, (tr, te) in enumerate(fold_splits):
    print("\n" + "=" * 100)
    print("HeteroGNN Fold:", fold)

    metrics, pred_df, hist_df = train_hetero_fold(
        train_idx=tr,
        test_idx=te,
        fold=fold,
        epochs=200,
        lr=7e-4,
        weight_decay=1e-4,
    )

    hetero_rows.append(metrics)
    hetero_preds.append(pred_df)
    hetero_histories.append(hist_df)

hetero_metrics = pd.DataFrame(hetero_rows)
hetero_predictions = pd.concat(hetero_preds, ignore_index=True)
hetero_history = pd.concat(hetero_histories, ignore_index=True)

display(hetero_metrics)

hetero_summary = (
    hetero_metrics
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(hetero_summary)

hetero_metrics.to_csv(GRAPH_DIR / "hetero_graphsage_fold_metrics.csv", index=False)
hetero_predictions.to_csv(GRAPH_DIR / "hetero_graphsage_fold_predictions.csv", index=False)
hetero_history.to_csv(GRAPH_DIR / "hetero_graphsage_training_history.csv", index=False)
hetero_summary.to_csv(GRAPH_DIR / "hetero_graphsage_summary.csv")

سلول ۳۳ — Hetero-GAT Encoder

In [ ]:
from torch_geometric.nn import GATConv, HeteroConv
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

class HeteroGATEncoder(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        out_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.lin_in = nn.Linear(in_dim, hidden_dim * heads)

        self.conv1 = HeteroConv(
            {
                ("protein", "enzyme_substrate", "protein"): GATConv(
                    (-1, -1),
                    hidden_dim,
                    heads=heads,
                    concat=True,
                    dropout=dropout,
                    add_self_loops=False,
                ),
                ("protein", "ppi", "protein"): GATConv(
                    (-1, -1),
                    hidden_dim,
                    heads=heads,
                    concat=True,
                    dropout=dropout,
                    add_self_loops=False,
                ),
                ("protein", "co_localized", "protein"): GATConv(
                    (-1, -1),
                    hidden_dim,
                    heads=heads,
                    concat=True,
                    dropout=dropout,
                    add_self_loops=False,
                ),
            },
            aggr="sum",
        )

        self.conv2 = HeteroConv(
            {
                ("protein", "enzyme_substrate", "protein"): GATConv(
                    (-1, -1),
                    out_dim,
                    heads=1,
                    concat=False,
                    dropout=dropout,
                    add_self_loops=False,
                ),
                ("protein", "ppi", "protein"): GATConv(
                    (-1, -1),
                    out_dim,
                    heads=1,
                    concat=False,
                    dropout=dropout,
                    add_self_loops=False,
                ),
                ("protein", "co_localized", "protein"): GATConv(
                    (-1, -1),
                    out_dim,
                    heads=1,
                    concat=False,
                    dropout=dropout,
                    add_self_loops=False,
                ),
            },
            aggr="sum",
        )

        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.norm2 = nn.LayerNorm(out_dim)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict):
        x = x_dict["protein"]

        x = self.lin_in(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.conv1(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.conv2(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm2(x)
        x = F.gelu(x)

        return {"protein": x}

سلول ۳۴ — مدل Hetero-GAT Link Predictor

In [ ]:
class HeteroGATLinkModel(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.encoder = HeteroGATEncoder(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=emb_dim,
            heads=heads,
            dropout=dropout,
        )

        self.predictor = HeteroLinkPredictor(
            emb_dim=emb_dim,
            hidden_dim=128,
            dropout=dropout,
        )

    def forward(self, data, edge_label_index):
        z_dict = self.encoder(
            data.x_dict,
            data.edge_index_dict,
        )

        logits = self.predictor(
            z_dict,
            edge_label_index,
        )

        return logits

سلول ۳۵ — Training loop برای Hetero-GAT

In [ ]:
def train_hetero_gat_fold(
    train_idx,
    test_idx,
    fold,
    epochs=200,
    lr=5e-4,
    weight_decay=1e-4,
):
    model = HeteroGATLinkModel(
        in_dim=hetero_data["protein"].x.shape[1],
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ).to(DEVICE)

    edge_label_index_all = hetero_data["protein", "predicts", "protein"].edge_label_index
    edge_label_all = hetero_data["protein", "predicts", "protein"].edge_label

    train_edge_index = edge_label_index_all[:, train_idx].to(DEVICE)
    train_y = edge_label_all[train_idx].to(DEVICE)

    test_edge_index = edge_label_index_all[:, test_idx].to(DEVICE)
    test_y = edge_label_all[test_idx].detach().cpu().numpy()

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_pr = -1
    best_state = None
    best_epoch = -1
    patience = 35
    bad_epochs = 0

    history = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(
            hetero_data,
            train_edge_index,
        )

        loss = criterion(logits, train_y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()

        model.eval()

        with torch.no_grad():
            test_logits = model(
                hetero_data,
                test_edge_index,
            )

            prob = torch.sigmoid(test_logits).detach().cpu().numpy()

        metrics = compute_metrics(test_y, prob)
        metrics["epoch"] = epoch
        metrics["train_loss"] = float(loss.detach().cpu().item())

        history.append(metrics)

        if metrics["pr_auc"] > best_pr:
            best_pr = metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 20 == 0:
            print(
                f"fold={fold} epoch={epoch} "
                f"loss={loss.item():.4f} "
                f"pr={metrics['pr_auc']:.4f} "
                f"roc={metrics['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping:", fold, epoch)
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        test_logits = model(
            hetero_data,
            test_edge_index,
        )
        prob = torch.sigmoid(test_logits).detach().cpu().numpy()

    final_metrics = compute_metrics(test_y, prob)
    final_metrics["fold"] = fold
    final_metrics["best_epoch"] = best_epoch

    pred_df = interaction_edges.iloc[test_idx][[
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "edge_label",
    ]].copy()

    pred_df["fold"] = fold
    pred_df["y_true"] = test_y
    pred_df["prob_hetero_gat"] = prob

    hist_df = pd.DataFrame(history)
    hist_df["fold"] = fold

    return final_metrics, pred_df, hist_df

سلول ۳۶ — اجرای Hetero-GAT

In [ ]:
hetero_gat_rows = []
hetero_gat_preds = []
hetero_gat_histories = []

for fold, (tr, te) in enumerate(fold_splits):
    print("\n" + "=" * 100)
    print("Hetero-GAT Fold:", fold)

    metrics, pred_df, hist_df = train_hetero_gat_fold(
        train_idx=tr,
        test_idx=te,
        fold=fold,
        epochs=200,
        lr=5e-4,
        weight_decay=1e-4,
    )

    hetero_gat_rows.append(metrics)
    hetero_gat_preds.append(pred_df)
    hetero_gat_histories.append(hist_df)

hetero_gat_metrics = pd.DataFrame(hetero_gat_rows)
hetero_gat_predictions = pd.concat(hetero_gat_preds, ignore_index=True)
hetero_gat_history = pd.concat(hetero_gat_histories, ignore_index=True)

display(hetero_gat_metrics)

hetero_gat_summary = (
    hetero_gat_metrics
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(hetero_gat_summary)

hetero_gat_metrics.to_csv(GRAPH_DIR / "hetero_gat_fold_metrics.csv", index=False)
hetero_gat_predictions.to_csv(GRAPH_DIR / "hetero_gat_fold_predictions.csv", index=False)
hetero_gat_history.to_csv(GRAPH_DIR / "hetero_gat_training_history.csv", index=False)
hetero_gat_summary.to_csv(GRAPH_DIR / "hetero_gat_summary.csv")

سلول ۳۷ — ساخت edge features برای HeteroData

In [ ]:
import torch
import numpy as np
import pandas as pd

from torch_geometric.data import HeteroData
from torch_geometric.nn import TransformerConv, HeteroConv

# ---------- edge feature helpers ----------

def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)

# compartment one-hot برای localization
compartments = sorted(loc_edges["compartment"].astype(str).dropna().unique())
comp_to_idx = {c: i for i, c in enumerate(compartments)}

base_dim = 6
edge_dim = base_dim + len(compartments)

print("n_compartments:", len(compartments))
print("edge_dim:", edge_dim)
print(compartments[:20])


def make_interaction_edge_attr(df):
    arr = np.zeros((len(df), edge_dim), dtype=np.float32)

    # 0: is_interaction_context
    arr[:, 0] = 1.0

    # 1: ppi flag
    if "ppi_physical_flag" in df.columns:
        arr[:, 1] = df["ppi_physical_flag"].astype(str).str.lower().isin(
            ["true", "1", "yes"]
        ).astype(float).values

    # 2: coloc flag
    if "coloc_flag" in df.columns:
        arr[:, 2] = df["coloc_flag"].astype(str).str.lower().isin(
            ["true", "1", "yes"]
        ).astype(float).values

    # 3: scrna crc
    if "scrna_coexpr_crc" in df.columns:
        arr[:, 3] = safe_numeric(df["scrna_coexpr_crc"]).values

    # 4: scrna lihc
    if "scrna_coexpr_lihc" in df.columns:
        arr[:, 4] = safe_numeric(df["scrna_coexpr_lihc"]).values

    # 5: scrna max-min expression
    if "scrna_max_min_expr" in df.columns:
        arr[:, 5] = safe_numeric(df["scrna_max_min_expr"]).values

    return torch.tensor(arr, dtype=torch.float32)


def make_ppi_edge_attr(df):
    arr = np.zeros((len(df), edge_dim), dtype=np.float32)

    # 1: ppi flag
    arr[:, 1] = 1.0

    return torch.tensor(arr, dtype=torch.float32)


def make_loc_edge_attr(df):
    arr = np.zeros((len(df), edge_dim), dtype=np.float32)

    # 2: coloc flag
    arr[:, 2] = 1.0

    for i, comp in enumerate(df["compartment"].astype(str).values):
        if comp in comp_to_idx:
            arr[i, base_dim + comp_to_idx[comp]] = 1.0

    return torch.tensor(arr, dtype=torch.float32)

سلول ۳۸ — ساخت HeteroData با edge_attr

In [ ]:
hetero_edge_data = HeteroData()

hetero_edge_data["protein"].x = torch.tensor(
    node_features,
    dtype=torch.float32
)

def edge_index_from_df(df):
    return torch.tensor(
        df[["src", "dst"]].values.T,
        dtype=torch.long
    )

# ---------- enzyme-substrate relation ----------
interaction_for_hetero = interaction_edges.copy()

interaction_edge_index = edge_index_from_df(interaction_for_hetero)
interaction_edge_attr = make_interaction_edge_attr(
    pairs.set_index("pair_id").loc[interaction_for_hetero["pair_id"]].reset_index()
    if "pair_id" in interaction_for_hetero.columns
    else interaction_for_hetero
)

hetero_edge_data["protein", "enzyme_substrate", "protein"].edge_index = interaction_edge_index
hetero_edge_data["protein", "enzyme_substrate", "protein"].edge_attr = interaction_edge_attr

# ---------- PPI relation ----------
ppi_edge_index = edge_index_from_df(ppi_edges)
ppi_edge_attr = make_ppi_edge_attr(ppi_edges)

ppi_edge_index = torch.cat(
    [ppi_edge_index, ppi_edge_index[[1, 0], :]],
    dim=1
)

ppi_edge_attr = torch.cat(
    [ppi_edge_attr, ppi_edge_attr],
    dim=0
)

hetero_edge_data["protein", "ppi", "protein"].edge_index = ppi_edge_index
hetero_edge_data["protein", "ppi", "protein"].edge_attr = ppi_edge_attr

# ---------- co-localization relation ----------
loc_edge_index = edge_index_from_df(loc_edges)
loc_edge_attr = make_loc_edge_attr(loc_edges)

loc_edge_index = torch.cat(
    [loc_edge_index, loc_edge_index[[1, 0], :]],
    dim=1
)

loc_edge_attr = torch.cat(
    [loc_edge_attr, loc_edge_attr],
    dim=0
)

hetero_edge_data["protein", "co_localized", "protein"].edge_index = loc_edge_index
hetero_edge_data["protein", "co_localized", "protein"].edge_attr = loc_edge_attr

# ---------- prediction edges ----------
hetero_edge_data["protein", "predicts", "protein"].edge_label_index = torch.tensor(
    interaction_edges[["src", "dst"]].values.T,
    dtype=torch.long
)

hetero_edge_data["protein", "predicts", "protein"].edge_label = torch.tensor(
    interaction_edges["edge_label"].astype(int).values,
    dtype=torch.float32
)

print(hetero_edge_data)
print("edge_attr enzyme_substrate:", hetero_edge_data["protein", "enzyme_substrate", "protein"].edge_attr.shape)
print("edge_attr ppi:", hetero_edge_data["protein", "ppi", "protein"].edge_attr.shape)
print("edge_attr loc:", hetero_edge_data["protein", "co_localized", "protein"].edge_attr.shape)

In [ ]:
pairs = pd.read_csv(
    "./Data_proc/pairs/pairs_all_embedding_ready.csv"
)

print(pairs.shape)
display(pairs.head())

In [ ]:
print("pair_id" in pairs.columns)
print("ppi_physical_flag" in pairs.columns)
print("coloc_flag" in pairs.columns)
print("scrna_coexpr_crc" in pairs.columns)
print("scrna_coexpr_lihc" in pairs.columns)
print("scrna_max_min_expr" in pairs.columns)

سلول ۳۹ — مدل Edge-Aware HeteroGNN

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class EdgeAwareHeteroEncoder(nn.Module):
    def __init__(
        self,
        in_dim,
        edge_dim,
        hidden_dim=128,
        out_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.lin_in = nn.Linear(in_dim, hidden_dim * heads)

        self.conv1 = HeteroConv(
            {
                ("protein", "enzyme_substrate", "protein"): TransformerConv(
                    (-1, -1),
                    hidden_dim,
                    heads=heads,
                    concat=True,
                    dropout=dropout,
                    edge_dim=edge_dim,
                ),
                ("protein", "ppi", "protein"): TransformerConv(
                    (-1, -1),
                    hidden_dim,
                    heads=heads,
                    concat=True,
                    dropout=dropout,
                    edge_dim=edge_dim,
                ),
                ("protein", "co_localized", "protein"): TransformerConv(
                    (-1, -1),
                    hidden_dim,
                    heads=heads,
                    concat=True,
                    dropout=dropout,
                    edge_dim=edge_dim,
                ),
            },
            aggr="sum",
        )

        self.conv2 = HeteroConv(
            {
                ("protein", "enzyme_substrate", "protein"): TransformerConv(
                    (-1, -1),
                    out_dim,
                    heads=1,
                    concat=False,
                    dropout=dropout,
                    edge_dim=edge_dim,
                ),
                ("protein", "ppi", "protein"): TransformerConv(
                    (-1, -1),
                    out_dim,
                    heads=1,
                    concat=False,
                    dropout=dropout,
                    edge_dim=edge_dim,
                ),
                ("protein", "co_localized", "protein"): TransformerConv(
                    (-1, -1),
                    out_dim,
                    heads=1,
                    concat=False,
                    dropout=dropout,
                    edge_dim=edge_dim,
                ),
            },
            aggr="sum",
        )

        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.norm2 = nn.LayerNorm(out_dim)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict, edge_attr_dict):
        x = x_dict["protein"]

        x = self.lin_in(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.conv1(
            x_dict,
            edge_index_dict,
            edge_attr_dict=edge_attr_dict,
        )

        x = x_dict["protein"]
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.conv2(
            x_dict,
            edge_index_dict,
            edge_attr_dict=edge_attr_dict,
        )

        x = x_dict["protein"]
        x = self.norm2(x)
        x = F.gelu(x)

        return {"protein": x}


class EdgeAwareHeteroLinkModel(nn.Module):
    def __init__(
        self,
        in_dim,
        edge_dim,
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.encoder = EdgeAwareHeteroEncoder(
            in_dim=in_dim,
            edge_dim=edge_dim,
            hidden_dim=hidden_dim,
            out_dim=emb_dim,
            heads=heads,
            dropout=dropout,
        )

        self.predictor = HeteroLinkPredictor(
            emb_dim=emb_dim,
            hidden_dim=128,
            dropout=dropout,
        )

    def forward(self, data, edge_label_index):
        z_dict = self.encoder(
            data.x_dict,
            data.edge_index_dict,
            data.edge_attr_dict,
        )

        logits = self.predictor(
            z_dict,
            edge_label_index,
        )

        return logits

سلول ۴۰ — انتقال به دستگاه

In [ ]:
def train_edge_aware_hetero_fold(
    train_idx,
    test_idx,
    fold,
    epochs=200,
    lr=5e-4,
    weight_decay=1e-4,
):
    model = EdgeAwareHeteroLinkModel(
        in_dim=hetero_edge_data["protein"].x.shape[1],
        edge_dim=edge_dim,
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ).to(DEVICE)

    edge_label_index_all = hetero_edge_data["protein", "predicts", "protein"].edge_label_index
    edge_label_all = hetero_edge_data["protein", "predicts", "protein"].edge_label

    train_edge_index = edge_label_index_all[:, train_idx].to(DEVICE)
    train_y = edge_label_all[train_idx].to(DEVICE)

    test_edge_index = edge_label_index_all[:, test_idx].to(DEVICE)
    test_y = edge_label_all[test_idx].detach().cpu().numpy()

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_pr = -1
    best_state = None
    best_epoch = -1
    patience = 35
    bad_epochs = 0

    history = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(
            hetero_edge_data,
            train_edge_index,
        )

        loss = criterion(logits, train_y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()

        model.eval()

        with torch.no_grad():
            test_logits = model(
                hetero_edge_data,
                test_edge_index,
            )

            prob = torch.sigmoid(test_logits).detach().cpu().numpy()

        metrics = compute_metrics(test_y, prob)
        metrics["epoch"] = epoch
        metrics["train_loss"] = float(loss.detach().cpu().item())

        history.append(metrics)

        if metrics["pr_auc"] > best_pr:
            best_pr = metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 20 == 0:
            print(
                f"fold={fold} epoch={epoch} "
                f"loss={loss.item():.4f} "
                f"pr={metrics['pr_auc']:.4f} "
                f"roc={metrics['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping:", fold, epoch)
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        test_logits = model(
            hetero_edge_data,
            test_edge_index,
        )
        prob = torch.sigmoid(test_logits).detach().cpu().numpy()

    final_metrics = compute_metrics(test_y, prob)
    final_metrics["fold"] = fold
    final_metrics["best_epoch"] = best_epoch

    pred_df = interaction_edges.iloc[test_idx][[
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "edge_label",
    ]].copy()

    pred_df["fold"] = fold
    pred_df["y_true"] = test_y
    pred_df["prob_edge_aware_hetero"] = prob

    hist_df = pd.DataFrame(history)
    hist_df["fold"] = fold

    return final_metrics, pred_df, hist_df

In [ ]:
hetero_edge_data = hetero_edge_data.to(DEVICE)

print("DEVICE:", DEVICE)
print("protein x device:", hetero_edge_data["protein"].x.device)

for edge_type in hetero_edge_data.edge_types:
    store = hetero_edge_data[edge_type]
    print(edge_type)
    
    if "edge_index" in store:
        print("  edge_index:", store.edge_index.device)
    
    if "edge_attr" in store:
        print("  edge_attr:", store.edge_attr.device)
    
    if "edge_label_index" in store:
        print("  edge_label_index:", store.edge_label_index.device)
    
    if "edge_label" in store:
        print("  edge_label:", store.edge_label.device)

سلول ۴۲ — اجرای Edge-Aware HeteroGNN

In [ ]:
edge_aware_rows = []
edge_aware_preds = []
edge_aware_histories = []

for fold, (tr, te) in enumerate(fold_splits):
    print("\n" + "=" * 100)
    print("Edge-Aware HeteroGNN Fold:", fold)

    metrics, pred_df, hist_df = train_edge_aware_hetero_fold(
        train_idx=tr,
        test_idx=te,
        fold=fold,
        epochs=200,
        lr=5e-4,
        weight_decay=1e-4,
    )

    edge_aware_rows.append(metrics)
    edge_aware_preds.append(pred_df)
    edge_aware_histories.append(hist_df)

edge_aware_hetero_metrics = pd.DataFrame(edge_aware_rows)
edge_aware_hetero_predictions = pd.concat(edge_aware_preds, ignore_index=True)
edge_aware_hetero_history = pd.concat(edge_aware_histories, ignore_index=True)

display(edge_aware_hetero_metrics)

edge_aware_hetero_summary = (
    edge_aware_hetero_metrics
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(edge_aware_hetero_summary)

edge_aware_hetero_metrics.to_csv(
    GRAPH_DIR / "edge_aware_hetero_fold_metrics.csv",
    index=False,
)

edge_aware_hetero_predictions.to_csv(
    GRAPH_DIR / "edge_aware_hetero_fold_predictions.csv",
    index=False,
)

edge_aware_hetero_history.to_csv(
    GRAPH_DIR / "edge_aware_hetero_training_history.csv",
    index=False,
)

edge_aware_hetero_summary.to_csv(
    GRAPH_DIR / "edge_aware_hetero_summary.csv"
)

سلول ۴۳ — تعریف HGT Model

In [ ]:
from torch_geometric.nn import HGTConv
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np


HGT_METADATA = (
    ["protein"],
    [
        ("protein", "enzyme_substrate", "protein"),
        ("protein", "ppi", "protein"),
        ("protein", "co_localized", "protein"),
    ],
)


class HGTEncoder(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        out_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.lin_in = nn.Linear(in_dim, hidden_dim)

        self.hgt1 = HGTConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            metadata=HGT_METADATA,
            heads=heads,
        )

        self.hgt2 = HGTConv(
            in_channels=hidden_dim,
            out_channels=out_dim,
            metadata=HGT_METADATA,
            heads=heads,
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(out_dim)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict):
        x = x_dict["protein"]

        x = self.lin_in(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.hgt1(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.hgt2(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm2(x)
        x = F.gelu(x)

        return {"protein": x}


class HGTLinkModel(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.encoder = HGTEncoder(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=emb_dim,
            heads=heads,
            dropout=dropout,
        )

        self.predictor = HeteroLinkPredictor(
            emb_dim=emb_dim,
            hidden_dim=128,
            dropout=dropout,
        )

    def forward(self, data, edge_label_index):
        edge_index_dict = {
            k: v for k, v in data.edge_index_dict.items()
            if k in HGT_METADATA[1]
        }

        z_dict = self.encoder(
            data.x_dict,
            edge_index_dict,
        )

        logits = self.predictor(
            z_dict,
            edge_label_index,
        )

        return logits

سلول ۴۴ — آماده‌سازی داده برای HGT

In [ ]:
hgt_data = hetero_data.to(DEVICE)

print(hgt_data)
print("protein x:", hgt_data["protein"].x.shape, hgt_data["protein"].x.device)

for etype in HGT_METADATA[1]:
    print(etype, hgt_data[etype].edge_index.shape, hgt_data[etype].edge_index.device)

سلول ۴۵ — Training loop برای HGT

In [ ]:
def train_hgt_fold(
    train_idx,
    test_idx,
    fold,
    epochs=200,
    lr=5e-4,
    weight_decay=1e-4,
):
    model = HGTLinkModel(
        in_dim=hgt_data["protein"].x.shape[1],
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ).to(DEVICE)

    edge_label_index_all = hgt_data["protein", "predicts", "protein"].edge_label_index
    edge_label_all = hgt_data["protein", "predicts", "protein"].edge_label

    train_edge_index = edge_label_index_all[:, train_idx].to(DEVICE)
    train_y = edge_label_all[train_idx].to(DEVICE)

    test_edge_index = edge_label_index_all[:, test_idx].to(DEVICE)
    test_y = edge_label_all[test_idx].detach().cpu().numpy()

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_pr = -1
    best_state = None
    best_epoch = -1
    patience = 35
    bad_epochs = 0

    history = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(
            hgt_data,
            train_edge_index,
        )

        loss = criterion(logits, train_y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()

        model.eval()

        with torch.no_grad():
            test_logits = model(
                hgt_data,
                test_edge_index,
            )
            prob = torch.sigmoid(test_logits).detach().cpu().numpy()

        metrics = compute_metrics(test_y, prob)
        metrics["epoch"] = epoch
        metrics["train_loss"] = float(loss.detach().cpu().item())

        history.append(metrics)

        if metrics["pr_auc"] > best_pr:
            best_pr = metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 20 == 0:
            print(
                f"fold={fold} epoch={epoch} "
                f"loss={loss.item():.4f} "
                f"pr={metrics['pr_auc']:.4f} "
                f"roc={metrics['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping:", fold, epoch)
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        test_logits = model(
            hgt_data,
            test_edge_index,
        )
        prob = torch.sigmoid(test_logits).detach().cpu().numpy()

    final_metrics = compute_metrics(test_y, prob)
    final_metrics["fold"] = fold
    final_metrics["best_epoch"] = best_epoch

    pred_df = interaction_edges.iloc[test_idx][[
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "edge_label",
    ]].copy()

    pred_df["fold"] = fold
    pred_df["y_true"] = test_y
    pred_df["prob_hgt"] = prob

    hist_df = pd.DataFrame(history)
    hist_df["fold"] = fold

    return final_metrics, pred_df, hist_df

سلول ۴۶ — اجرای HGT

In [ ]:
hgt_rows = []
hgt_preds = []
hgt_histories = []

for fold, (tr, te) in enumerate(fold_splits):
    print("\n" + "=" * 100)
    print("HGT Fold:", fold)

    metrics, pred_df, hist_df = train_hgt_fold(
        train_idx=tr,
        test_idx=te,
        fold=fold,
        epochs=200,
        lr=5e-4,
        weight_decay=1e-4,
    )

    hgt_rows.append(metrics)
    hgt_preds.append(pred_df)
    hgt_histories.append(hist_df)

hgt_metrics = pd.DataFrame(hgt_rows)
hgt_predictions = pd.concat(hgt_preds, ignore_index=True)
hgt_history = pd.concat(hgt_histories, ignore_index=True)

display(hgt_metrics)

hgt_summary = (
    hgt_metrics
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(hgt_summary)

hgt_metrics.to_csv(GRAPH_DIR / "hgt_fold_metrics.csv", index=False)
hgt_predictions.to_csv(GRAPH_DIR / "hgt_fold_predictions.csv", index=False)
hgt_history.to_csv(GRAPH_DIR / "hgt_training_history.csv", index=False)
hgt_summary.to_csv(GRAPH_DIR / "hgt_summary.csv")

سلول ۴۷ — لود همه predictionها

In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    brier_score_loss,
)

GRAPH_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset"
NN_RESULT_DIR = PROJECT_ROOT / "Data_ml" / "neural_pair_scorer_results"

hgt_pred = pd.read_csv(GRAPH_DIR / "hgt_fold_predictions.csv", dtype=str, low_memory=False)
edgeaware_pred = pd.read_csv(GRAPH_DIR / "edge_aware_hetero_fold_predictions.csv", dtype=str, low_memory=False)
hetero_pred = pd.read_csv(GRAPH_DIR / "hetero_graphsage_fold_predictions.csv", dtype=str, low_memory=False)
graphsage_pred = pd.read_csv(GRAPH_DIR / "graphsage_fold_predictions.csv", dtype=str, low_memory=False)

xgb_pred = pd.read_csv(
    NN_RESULT_DIR / "xgb_best_fusion_fold_predictions_for_ensemble.csv",
    dtype=str,
    low_memory=False,
)

nn_pred = pd.read_csv(
    NN_RESULT_DIR / "nn_focal_fold_predictions.csv",
    dtype=str,
    low_memory=False,
)

print("hgt:", hgt_pred.shape)
print("edgeaware:", edgeaware_pred.shape)
print("hetero:", hetero_pred.shape)
print("graphsage:", graphsage_pred.shape)
print("xgb:", xgb_pred.shape)
print("nn:", nn_pred.shape)

سلول ۴۸ — تمیزسازی ستون‌ها

In [ ]:
def clean_pred_df(df, prob_col, new_prob_col):
    df = df.copy()
    
    df["fold"] = (
        pd.to_numeric(df["fold"], errors="coerce")
        .astype("Int64")
        .astype(int)
    )
    
    if "y_true" in df.columns:
        df["y_true"] = pd.to_numeric(df["y_true"], errors="coerce").astype(int)
    
    df[new_prob_col] = pd.to_numeric(df[prob_col], errors="coerce")
    
    return df


hgt_pred = clean_pred_df(hgt_pred, "prob_hgt", "prob_hgt")
edgeaware_pred = clean_pred_df(edgeaware_pred, "prob_edge_aware_hetero", "prob_edgeaware")
hetero_pred = clean_pred_df(hetero_pred, "prob_hetero_graphsage", "prob_hetero")
graphsage_pred = clean_pred_df(graphsage_pred, "prob_graphsage", "prob_graphsage")
xgb_pred = clean_pred_df(xgb_pred, "prob_xgb", "prob_xgb")
nn_pred = clean_pred_df(nn_pred, "prob", "prob_nn")

سلول ۴۹ — Merge فقط روی pair_id

چون foldها بین مدل‌ها ممکن است متفاوت باشند، fold مرجع را از HGT می‌گیریم.

In [ ]:
mega_df = hgt_pred[[
    "pair_id",
    "group_id",
    "enzyme_class",
    "enz_ac",
    "sub_ac",
    "enz_gene",
    "sub_gene",
    "edge_label",
    "fold",
    "y_true",
    "prob_hgt",
]].copy()

mega_df = mega_df.merge(
    edgeaware_pred[["pair_id", "prob_edgeaware"]],
    on="pair_id",
    how="inner",
)

mega_df = mega_df.merge(
    hetero_pred[["pair_id", "prob_hetero"]],
    on="pair_id",
    how="inner",
)

mega_df = mega_df.merge(
    graphsage_pred[["pair_id", "prob_graphsage"]],
    on="pair_id",
    how="inner",
)

mega_df = mega_df.merge(
    xgb_pred[["pair_id", "prob_xgb"]],
    on="pair_id",
    how="inner",
)

mega_df = mega_df.merge(
    nn_pred[["pair_id", "prob_nn"]],
    on="pair_id",
    how="inner",
)

print("mega_df:", mega_df.shape)
print("unique pair_id:", mega_df["pair_id"].nunique())

print("missing:")
print(
    mega_df[
        [
            "prob_hgt",
            "prob_edgeaware",
            "prob_hetero",
            "prob_graphsage",
            "prob_xgb",
            "prob_nn",
            "y_true",
        ]
    ].isna().sum()
)

assert mega_df.shape[0] == 6196
assert mega_df["pair_id"].nunique() == 6196

display(mega_df.head())

سلول ۵۰ — تابع ارزیابی

In [ ]:
def evaluate_probs_by_fold(df, prob_col):
    rows = []

    for fold, g in df.groupby("fold"):
        y_true = g["y_true"].astype(int).values
        prob = g[prob_col].astype(float).values
        pred = (prob >= 0.5).astype(int)

        rows.append({
            "fold": fold,
            "roc_auc": roc_auc_score(y_true, prob),
            "pr_auc": average_precision_score(y_true, prob),
            "f1": f1_score(y_true, pred, zero_division=0),
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "brier": brier_score_loss(y_true, prob),
        })

    return pd.DataFrame(rows)

سلول ۵۱ — Weighted Mega Ensemble

In [ ]:
mega_weight_sets = [
    {
        "name": "hgt_only",
        "w_hgt": 1.0,
        "w_edgeaware": 0.0,
        "w_hetero": 0.0,
        "w_graphsage": 0.0,
        "w_xgb": 0.0,
        "w_nn": 0.0,
    },
    {
        "name": "hgt0.7_edge0.3",
        "w_hgt": 0.7,
        "w_edgeaware": 0.3,
        "w_hetero": 0.0,
        "w_graphsage": 0.0,
        "w_xgb": 0.0,
        "w_nn": 0.0,
    },
    {
        "name": "hgt0.6_edge0.4",
        "w_hgt": 0.6,
        "w_edgeaware": 0.4,
        "w_hetero": 0.0,
        "w_graphsage": 0.0,
        "w_xgb": 0.0,
        "w_nn": 0.0,
    },
    {
        "name": "hgt0.5_edge0.3_hetero0.2",
        "w_hgt": 0.5,
        "w_edgeaware": 0.3,
        "w_hetero": 0.2,
        "w_graphsage": 0.0,
        "w_xgb": 0.0,
        "w_nn": 0.0,
    },
    {
        "name": "hgt0.5_edge0.3_graph0.2",
        "w_hgt": 0.5,
        "w_edgeaware": 0.3,
        "w_hetero": 0.0,
        "w_graphsage": 0.2,
        "w_xgb": 0.0,
        "w_nn": 0.0,
    },
    {
        "name": "hgt0.5_edge0.2_xgb0.2_nn0.1",
        "w_hgt": 0.5,
        "w_edgeaware": 0.2,
        "w_hetero": 0.0,
        "w_graphsage": 0.0,
        "w_xgb": 0.2,
        "w_nn": 0.1,
    },
    {
        "name": "hgt0.4_edge0.2_hetero0.2_xgb0.1_nn0.1",
        "w_hgt": 0.4,
        "w_edgeaware": 0.2,
        "w_hetero": 0.2,
        "w_graphsage": 0.0,
        "w_xgb": 0.1,
        "w_nn": 0.1,
    },
    {
        "name": "all_equal",
        "w_hgt": 1/6,
        "w_edgeaware": 1/6,
        "w_hetero": 1/6,
        "w_graphsage": 1/6,
        "w_xgb": 1/6,
        "w_nn": 1/6,
    },
]

mega_weighted_rows = []
mega_weighted_fold_tables = []

for ws in mega_weight_sets:
    col = "prob_" + ws["name"]

    mega_df[col] = (
        ws["w_hgt"] * mega_df["prob_hgt"]
        + ws["w_edgeaware"] * mega_df["prob_edgeaware"]
        + ws["w_hetero"] * mega_df["prob_hetero"]
        + ws["w_graphsage"] * mega_df["prob_graphsage"]
        + ws["w_xgb"] * mega_df["prob_xgb"]
        + ws["w_nn"] * mega_df["prob_nn"]
    )

    fold_metrics = evaluate_probs_by_fold(mega_df, col)
    fold_metrics["model"] = ws["name"]

    for k, v in ws.items():
        if k != "name":
            fold_metrics[k] = v

    mega_weighted_fold_tables.append(fold_metrics)

    row = {"model": ws["name"]}
    for k, v in ws.items():
        if k != "name":
            row[k] = v

    for m in ["roc_auc", "pr_auc", "f1", "accuracy", "precision", "recall", "brier"]:
        row[f"{m}_mean"] = fold_metrics[m].mean()
        row[f"{m}_std"] = fold_metrics[m].std()

    mega_weighted_rows.append(row)

mega_weighted_results = pd.DataFrame(mega_weighted_rows).sort_values(
    "pr_auc_mean",
    ascending=False,
)

mega_weighted_fold_results = pd.concat(
    mega_weighted_fold_tables,
    ignore_index=True,
)

display(mega_weighted_results)

mega_weighted_results.to_csv(
    GRAPH_DIR / "mega_weighted_ensemble_results.csv",
    index=False,
)

mega_weighted_fold_results.to_csv(
    GRAPH_DIR / "mega_weighted_ensemble_fold_results.csv",
    index=False,
)

سلول ۵۲ — Logistic Stacking Mega Ensemble

In [ ]:
stack_features = [
    "prob_hgt",
    "prob_edgeaware",
    "prob_hetero",
    "prob_graphsage",
    "prob_xgb",
    "prob_nn",
]

mega_stack_rows = []
mega_stack_pred_rows = []

for test_fold in sorted(mega_df["fold"].unique()):
    train_stack = mega_df[mega_df["fold"] != test_fold].copy()
    test_stack = mega_df[mega_df["fold"] == test_fold].copy()

    X_train_stack = train_stack[stack_features].values
    y_train_stack = train_stack["y_true"].astype(int).values

    X_test_stack = test_stack[stack_features].values
    y_test_stack = test_stack["y_true"].astype(int).values

    meta_model = LogisticRegression(
        solver="liblinear",
        random_state=42,
    )

    meta_model.fit(X_train_stack, y_train_stack)

    prob_stack = meta_model.predict_proba(X_test_stack)[:, 1]
    pred_stack = (prob_stack >= 0.5).astype(int)

    row = {
        "fold": test_fold,
        "roc_auc": roc_auc_score(y_test_stack, prob_stack),
        "pr_auc": average_precision_score(y_test_stack, prob_stack),
        "f1": f1_score(y_test_stack, pred_stack, zero_division=0),
        "accuracy": accuracy_score(y_test_stack, pred_stack),
        "precision": precision_score(y_test_stack, pred_stack, zero_division=0),
        "recall": recall_score(y_test_stack, pred_stack, zero_division=0),
        "brier": brier_score_loss(y_test_stack, prob_stack),
        "intercept": meta_model.intercept_[0],
    }

    for i, feat in enumerate(stack_features):
        row["coef_" + feat] = meta_model.coef_[0][i]

    mega_stack_rows.append(row)

    tmp = test_stack[[
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "edge_label",
        "fold",
        "y_true",
    ] + stack_features].copy()

    tmp["prob_mega_stack"] = prob_stack
    mega_stack_pred_rows.append(tmp)

mega_stack_metrics = pd.DataFrame(mega_stack_rows)
mega_stack_predictions = pd.concat(mega_stack_pred_rows, ignore_index=True)

display(mega_stack_metrics)

mega_stack_summary = (
    mega_stack_metrics
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(mega_stack_summary)

mega_stack_metrics.to_csv(
    GRAPH_DIR / "mega_stacking_fold_metrics.csv",
    index=False,
)

mega_stack_predictions.to_csv(
    GRAPH_DIR / "mega_stacking_predictions.csv",
    index=False,
)

mega_stack_summary.to_csv(
    GRAPH_DIR / "mega_stacking_summary.csv"
)

سلول ۵۳ — جدول نهایی مدل‌های گرافی

In [ ]:
final_graph_rows = []

summary_paths = [
    ("GraphSAGE", GRAPH_DIR / "graphsage_fold_metrics.csv"),
    ("GAT", GRAPH_DIR / "gat_fold_metrics.csv"),
    ("HeteroGraphSAGE", GRAPH_DIR / "hetero_graphsage_fold_metrics.csv"),
    ("HeteroGAT", GRAPH_DIR / "hetero_gat_fold_metrics.csv"),
    ("EdgeAwareHeteroGNN", GRAPH_DIR / "edge_aware_hetero_fold_metrics.csv"),
    ("HGT", GRAPH_DIR / "hgt_fold_metrics.csv"),
]

for name, path in summary_paths:
    if path.exists():
        df = pd.read_csv(path)
        row = {"model": name}
        for m in ["roc_auc", "pr_auc", "f1", "accuracy", "precision", "recall", "brier"]:
            row[f"{m}_mean"] = df[m].mean()
            row[f"{m}_std"] = df[m].std()
        final_graph_rows.append(row)

for _, r in mega_weighted_results.iterrows():
    row = {"model": "MegaWeighted_" + r["model"]}
    for m in ["roc_auc", "pr_auc", "f1", "accuracy", "precision", "recall", "brier"]:
        row[f"{m}_mean"] = r[f"{m}_mean"]
        row[f"{m}_std"] = r[f"{m}_std"]
    final_graph_rows.append(row)

row = {"model": "MegaStacking_LogReg"}
for m in ["roc_auc", "pr_auc", "f1", "accuracy", "precision", "recall", "brier"]:
    row[f"{m}_mean"] = mega_stack_metrics[m].mean()
    row[f"{m}_std"] = mega_stack_metrics[m].std()
final_graph_rows.append(row)

day12_final_graph_leaderboard = pd.DataFrame(final_graph_rows).sort_values(
    "pr_auc_mean",
    ascending=False,
)

display(day12_final_graph_leaderboard)

day12_final_graph_leaderboard.to_csv(
    GRAPH_DIR / "day12_final_graph_leaderboard.csv",
    index=False,
)

سلول ۱ — مسیرها و load داده‌ها

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    brier_score_loss,
)

from sklearn.model_selection import GroupKFold
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv


PROJECT_ROOT = Path(".")
GRAPH_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset"

EXPLAIN_DIR = GRAPH_DIR / "day13_explainability"
MODEL_DIR = EXPLAIN_DIR / "saved_models"

EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

nodes = pd.read_csv(GRAPH_DIR / "graph_nodes.csv")
interaction_edges = pd.read_csv(GRAPH_DIR / "interaction_edges_labeled.csv")
ppi_edges = pd.read_csv(GRAPH_DIR / "ppi_edges.csv")
loc_edges = pd.read_csv(GRAPH_DIR / "colocalization_edges.csv")
node_features = np.load(GRAPH_DIR / "node_features_esm650.npy")

print("nodes:", nodes.shape)
print("interaction_edges:", interaction_edges.shape)
print("ppi_edges:", ppi_edges.shape)
print("loc_edges:", loc_edges.shape)
print("node_features:", node_features.shape)

سلول ۲ — ساخت HeteroData برای HGT

In [ ]:
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("DEVICE:", DEVICE)

hetero_data = HeteroData()
hetero_data["protein"].x = torch.tensor(node_features, dtype=torch.float32)

def edge_index_from_df(df):
    return torch.tensor(df[["src", "dst"]].values.T, dtype=torch.long)

# enzyme-substrate
hetero_data["protein", "enzyme_substrate", "protein"].edge_index = edge_index_from_df(
    interaction_edges
)

# ppi, undirected
ppi_edge_index = edge_index_from_df(ppi_edges)
ppi_edge_index = torch.cat([ppi_edge_index, ppi_edge_index[[1, 0], :]], dim=1)
hetero_data["protein", "ppi", "protein"].edge_index = ppi_edge_index

# co-localization, undirected
loc_edge_index = edge_index_from_df(loc_edges)
loc_edge_index = torch.cat([loc_edge_index, loc_edge_index[[1, 0], :]], dim=1)
hetero_data["protein", "co_localized", "protein"].edge_index = loc_edge_index

# labeled edges for prediction
hetero_data["protein", "predicts", "protein"].edge_label_index = torch.tensor(
    interaction_edges[["src", "dst"]].values.T,
    dtype=torch.long,
)

hetero_data["protein", "predicts", "protein"].edge_label = torch.tensor(
    interaction_edges["edge_label"].astype(int).values,
    dtype=torch.float32,
)

hetero_data = hetero_data.to(DEVICE)

print(hetero_data)

سلول ۳ — ساخت foldها مثل قبل

In [ ]:
y = interaction_edges["edge_label"].astype(int).values
groups = interaction_edges["group_id"].astype(str).values

gkf = GroupKFold(n_splits=5)
fold_splits = []

for fold, (tr, te) in enumerate(gkf.split(interaction_edges, y, groups)):
    fold_splits.append((tr, te))
    print(
        fold,
        "train:", len(tr),
        "test:", len(te),
        "test_pos:", y[te].sum(),
        "test_neg:", len(te) - y[te].sum(),
    )

سلول ۴ — تابع metric

In [ ]:
def compute_metrics(y_true, prob):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    pred = (prob >= 0.5).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, prob),
        "pr_auc": average_precision_score(y_true, prob),
        "f1": f1_score(y_true, pred, zero_division=0),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "brier": brier_score_loss(y_true, prob),
    }

سلول ۵ — تعریف HGT Model

In [ ]:
HGT_METADATA = (
    ["protein"],
    [
        ("protein", "enzyme_substrate", "protein"),
        ("protein", "ppi", "protein"),
        ("protein", "co_localized", "protein"),
    ],
)


class HGTEncoder(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        out_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.lin_in = nn.Linear(in_dim, hidden_dim)

        self.hgt1 = HGTConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            metadata=HGT_METADATA,
            heads=heads,
        )

        self.hgt2 = HGTConv(
            in_channels=hidden_dim,
            out_channels=out_dim,
            metadata=HGT_METADATA,
            heads=heads,
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(out_dim)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict):
        x = x_dict["protein"]

        x = self.lin_in(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.hgt1(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.hgt2(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm2(x)
        x = F.gelu(x)

        return {"protein": x}


class HeteroLinkPredictor(nn.Module):
    def __init__(self, emb_dim=128, hidden_dim=128, dropout=0.35):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, z_dict, edge_label_index):
        z = z_dict["protein"]

        src = edge_label_index[0]
        dst = edge_label_index[1]

        z_src = z[src]
        z_dst = z[dst]

        h = torch.cat(
            [
                z_src,
                z_dst,
                torch.abs(z_src - z_dst),
                z_src * z_dst,
            ],
            dim=1,
        )

        return self.mlp(h).squeeze(-1)


class HGTLinkModel(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.encoder = HGTEncoder(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=emb_dim,
            heads=heads,
            dropout=dropout,
        )

        self.predictor = HeteroLinkPredictor(
            emb_dim=emb_dim,
            hidden_dim=128,
            dropout=dropout,
        )

    def forward(self, data, edge_label_index):
        edge_index_dict = {
            k: v
            for k, v in data.edge_index_dict.items()
            if k in HGT_METADATA[1]
        }

        z_dict = self.encoder(
            data.x_dict,
            edge_index_dict,
        )

        logits = self.predictor(
            z_dict,
            edge_label_index,
        )

        return logits

سلول ۶ — train یک fold و ذخیره best model

In [ ]:
def train_hgt_fold_save(
    train_idx,
    test_idx,
    fold,
    epochs=200,
    lr=5e-4,
    weight_decay=1e-4,
):
    model = HGTLinkModel(
        in_dim=hetero_data["protein"].x.shape[1],
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ).to(DEVICE)

    edge_label_index_all = hetero_data["protein", "predicts", "protein"].edge_label_index
    edge_label_all = hetero_data["protein", "predicts", "protein"].edge_label

    train_edge_index = edge_label_index_all[:, train_idx].to(DEVICE)
    train_y = edge_label_all[train_idx].to(DEVICE)

    test_edge_index = edge_label_index_all[:, test_idx].to(DEVICE)
    test_y = edge_label_all[test_idx].detach().cpu().numpy()

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_pr = -1
    best_state = None
    best_epoch = -1
    patience = 35
    bad_epochs = 0
    history = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(hetero_data, train_edge_index)
        loss = criterion(logits, train_y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()

        model.eval()

        with torch.no_grad():
            test_logits = model(hetero_data, test_edge_index)
            prob = torch.sigmoid(test_logits).detach().cpu().numpy()

        metrics = compute_metrics(test_y, prob)
        metrics["epoch"] = epoch
        metrics["train_loss"] = float(loss.detach().cpu().item())
        history.append(metrics)

        if metrics["pr_auc"] > best_pr:
            best_pr = metrics["pr_auc"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 20 == 0:
            print(
                f"fold={fold} epoch={epoch} "
                f"loss={loss.item():.4f} "
                f"pr={metrics['pr_auc']:.4f} "
                f"roc={metrics['roc_auc']:.4f}"
            )

        if bad_epochs >= patience:
            print("Early stopping:", fold, epoch)
            break

    model.load_state_dict(best_state)
    model.eval()

    model_path = MODEL_DIR / f"hgt_fold{fold}_best.pt"

    torch.save(
        {
            "fold": fold,
            "best_epoch": best_epoch,
            "model_state_dict": best_state,
            "in_dim": hetero_data["protein"].x.shape[1],
            "hidden_dim": 128,
            "emb_dim": 128,
            "heads": 4,
            "dropout": 0.35,
            "metadata": HGT_METADATA,
        },
        model_path,
    )

    with torch.no_grad():
        test_logits = model(hetero_data, test_edge_index)
        prob = torch.sigmoid(test_logits).detach().cpu().numpy()

    final_metrics = compute_metrics(test_y, prob)
    final_metrics["fold"] = fold
    final_metrics["best_epoch"] = best_epoch
    final_metrics["model_path"] = str(model_path)

    pred_df = interaction_edges.iloc[test_idx][[
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "edge_label",
    ]].copy()

    pred_df["fold"] = fold
    pred_df["y_true"] = test_y
    pred_df["prob_hgt_saved"] = prob

    hist_df = pd.DataFrame(history)
    hist_df["fold"] = fold

    return final_metrics, pred_df, hist_df

سلول ۷ — اجرای ۵ fold و ذخیره خروجی‌ها

In [ ]:
hgt_saved_rows = []
hgt_saved_preds = []
hgt_saved_histories = []

for fold, (tr, te) in enumerate(fold_splits):
    print("\n" + "=" * 100)
    print("Saving HGT Fold:", fold)

    metrics, pred_df, hist_df = train_hgt_fold_save(
        train_idx=tr,
        test_idx=te,
        fold=fold,
        epochs=200,
        lr=5e-4,
        weight_decay=1e-4,
    )

    hgt_saved_rows.append(metrics)
    hgt_saved_preds.append(pred_df)
    hgt_saved_histories.append(hist_df)

hgt_saved_metrics = pd.DataFrame(hgt_saved_rows)
hgt_saved_predictions = pd.concat(hgt_saved_preds, ignore_index=True)
hgt_saved_history = pd.concat(hgt_saved_histories, ignore_index=True)

display(hgt_saved_metrics)

hgt_saved_summary = (
    hgt_saved_metrics
    .drop(columns=["fold"])
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
)

display(hgt_saved_summary)

hgt_saved_metrics.to_csv(EXPLAIN_DIR / "hgt_saved_fold_metrics.csv", index=False)
hgt_saved_predictions.to_csv(EXPLAIN_DIR / "hgt_saved_fold_predictions.csv", index=False)
hgt_saved_history.to_csv(EXPLAIN_DIR / "hgt_saved_training_history.csv", index=False)
hgt_saved_summary.to_csv(EXPLAIN_DIR / "hgt_saved_summary.csv")

سلول ۸ — چک ذخیره مدل‌ها

In [ ]:
saved_models = sorted(MODEL_DIR.glob("hgt_fold*_best.pt"))

print("n saved models:", len(saved_models))

for f in saved_models:
    print(f)